In [1]:
"""
build_data.py — Run once locally to generate all CSVs for the GitHub Pages site.
Requires internet access to data.cityofchicago.org.
Output: 6 CSVs in the current directory.
"""
import pandas as pd
from urllib.parse import urlencode

# ============================================================
# PRIMARY DATASET — Chicago Food Inspections
# ============================================================
URL = "https://data.cityofchicago.org/api/views/4ijn-s7e5/rows.csv?accessType=DOWNLOAD"
print("Loading Food Inspections dataset (~323 MB, may take a minute)...")
df = pd.read_csv(URL)
print(f"Raw shape: {df.shape}")

df['Inspection Date'] = pd.to_datetime(df['Inspection Date'])
df['Year'] = df['Inspection Date'].dt.year

df_filtered = df[
    df['Results'].isin(['Pass', 'Fail', 'Pass w/ Conditions']) &
    df['Risk'].isin(['Risk 1 (High)', 'Risk 2 (Medium)', 'Risk 3 (Low)']) &
    df['Facility Type'].notna() &
    (df['Year'] >= 2010) & (df['Year'] <= 2025)
].copy()

top_facilities = df_filtered['Facility Type'].value_counts().head(8).index.tolist()
df_filtered = df_filtered[df_filtered['Facility Type'].isin(top_facilities)]
print(f"Filtered shape: {df_filtered.shape}")

# ---- 1. driver_data.csv (24 rows) ----
driver_data = (
    df_filtered.groupby(['Facility Type', 'Results']).size().reset_index(name='Count')
)
driver_data['Total per Facility'] = driver_data.groupby('Facility Type')['Count'].transform('sum')
driver_data['Percentage'] = (driver_data['Count'] / driver_data['Total per Facility'] * 100).round(1)
driver_data.to_csv('driver_data.csv', index=False)
print(f"✓ driver_data.csv: {driver_data.shape}")

# ---- 2. yearly_pass.csv (128 rows) ----
yearly_total = df_filtered.groupby(['Facility Type', 'Year']).size().reset_index(name='Total')
yearly_pass_count = (
    df_filtered[df_filtered['Results'] == 'Pass']
    .groupby(['Facility Type', 'Year']).size().reset_index(name='Count')
)
yearly_pass = yearly_pass_count.merge(yearly_total, on=['Facility Type', 'Year'])
yearly_pass['Pass Rate'] = (yearly_pass['Count'] / yearly_pass['Total'] * 100).round(2)
yearly_pass = yearly_pass[['Facility Type', 'Year', 'Pass Rate', 'Total']]
yearly_pass.to_csv('yearly_pass.csv', index=False)
print(f"✓ yearly_pass.csv: {yearly_pass.shape}")

# ---- 3. before_after.csv (8 rows) ----
before = yearly_pass[yearly_pass['Year'] == 2017][['Facility Type', 'Pass Rate']].rename(
    columns={'Pass Rate': 'Pass_2017'}
)
after = yearly_pass[yearly_pass['Year'] == 2019][['Facility Type', 'Pass Rate']].rename(
    columns={'Pass Rate': 'Pass_2019'}
)
before_after = before.merge(after, on='Facility Type')
before_after['Drop'] = (before_after['Pass_2017'] - before_after['Pass_2019']).round(2)
before_after = before_after.sort_values('Drop', ascending=False)
before_after.to_csv('before_after.csv', index=False)
print(f"✓ before_after.csv: {before_after.shape}")

# ---- 4. yearly_all_results.csv ----
yearly_all = (
    df_filtered.groupby(['Facility Type', 'Year', 'Results']).size().reset_index(name='Count')
)
yearly_facility_total = yearly_all.groupby(['Facility Type', 'Year'])['Count'].transform('sum')
yearly_all['Percentage'] = (yearly_all['Count'] / yearly_facility_total * 100).round(2)
yearly_all.to_csv('yearly_all_results.csv', index=False)
print(f"✓ yearly_all_results.csv: {yearly_all.shape}")

# ============================================================
# CONTEXTUAL DATASET — Chicago Business Licenses
# Pull license_number, description, start, expiration via Socrata API
# ============================================================
print("\nQuerying Business Licenses dataset (server-side filtered)...")

food_categories = [
    "Retail Food Establishment",
    "Wholesale Food Establishment",
    "Mobile Food Dispenser",
    "Mobile Food License",
    "Mobile Frozen Desserts Dispenser - Non-Motorized",
    "Food - Shared Kitchen",
    "Food - Shared Kitchen - Supplemental",
    "Pop-Up Food Est. User - Tier I",
    "Pop-Up Food Est. User - Tier II",
    "Pop-Up Food Est. User - Tier III",
    "Peddler, food (fruits and vegtables only)",
    "Peddler,food - (fruits and vegetables only) - special",
    "Navy Pier Vendor (Food)",
    "Navy Pier Vendor (Food) 30 Day",
]

def soql_escape(s):
    return s.replace("'", "''")

in_clause = ", ".join(f"'{soql_escape(c)}'" for c in food_categories)

params = {
    "$select": "license_number,license_description,license_start_date,expiration_date",
    "$where": f"license_description in({in_clause})",
    "$limit": "500000",
}
LIC_API = "https://data.cityofchicago.org/resource/r5kz-chrr.csv?" + urlencode(params)

food_lic = pd.read_csv(LIC_API)
print(f"Food-related license rows downloaded: {food_lic.shape}")

food_lic['license_start_date'] = pd.to_datetime(food_lic['license_start_date'], errors='coerce')
food_lic['expiration_date']    = pd.to_datetime(food_lic['expiration_date'],    errors='coerce')
food_lic['Year']               = food_lic['license_start_date'].dt.year

# ============================================================
# Categorize licenses into clean buckets — used by both Coverage and Option 5
# ============================================================
def categorize_license(desc):
    if pd.isna(desc):
        return 'Other'
    d = str(desc).lower()
    if 'retail food' in d:
        return 'Retail Food (restaurants, grocery, bakery)'
    if 'wholesale' in d:
        return 'Wholesale Food'
    if 'mobile' in d:
        return 'Mobile Food (trucks & carts)'
    if 'pop-up' in d or 'pop up' in d:
        return 'Pop-Up Food'
    if 'shared kitchen' in d:
        return 'Shared Kitchen'
    if 'peddler' in d:
        return 'Food Peddler'
    if 'navy pier' in d:
        return 'Navy Pier Food Vendor'
    return 'Other'

food_lic['Category'] = food_lic['license_description'].apply(categorize_license)

# ============================================================
# Compute "active food licenses on July 1 of each year"
# ============================================================
years = list(range(2010, 2026))
active_per_year = []
for year in years:
    snapshot = pd.Timestamp(year=year, month=7, day=1)
    active_mask = (
        (food_lic['license_start_date'] <= snapshot) &
        (food_lic['expiration_date']    >= snapshot)
    )
    n_active = food_lic.loc[active_mask, 'license_number'].nunique()
    active_per_year.append({'Year': year, 'Active Licenses': n_active})
active_licenses = pd.DataFrame(active_per_year)

# ============================================================
# Compute "unique establishments inspected per year"
# ============================================================
df_for_join = df.copy()
df_for_join['Year'] = df_for_join['Inspection Date'].dt.year
df_for_join = df_for_join[(df_for_join['Year'] >= 2010) & (df_for_join['Year'] <= 2025)]
df_for_join['License #'] = pd.to_numeric(df_for_join['License #'], errors='coerce')
df_for_join = df_for_join[df_for_join['License #'] > 0]

inspected_per_year = (
    df_for_join.groupby('Year')['License #'].nunique().reset_index(name='Establishments Inspected')
)

# ---- 5. coverage_gap.csv (16 rows) ----
coverage = active_licenses.merge(inspected_per_year, on='Year', how='outer').fillna(0)
coverage['Active Licenses']          = coverage['Active Licenses'].astype(int)
coverage['Establishments Inspected'] = coverage['Establishments Inspected'].astype(int)
coverage['Uninspected'] = coverage['Active Licenses'] - coverage['Establishments Inspected']
coverage['Coverage %']  = (
    coverage['Establishments Inspected'] / coverage['Active Licenses'] * 100
).round(1)
coverage = coverage.sort_values('Year').reset_index(drop=True)
coverage.to_csv('coverage_gap.csv', index=False)
print(f"✓ coverage_gap.csv: {coverage.shape}")

# ============================================================
# OPTION 5 — Inspection outcomes by license category
# Joins inspections to licenses by license_number
# ============================================================
print("\nJoining inspections to licenses by License Number for Option 5...")

df_join = df.copy()
df_join['Year'] = df_join['Inspection Date'].dt.year
df_join['License #'] = pd.to_numeric(df_join['License #'], errors='coerce')
df_join = df_join[
    (df_join['Year'] >= 2010) & (df_join['Year'] <= 2025) &
    df_join['Results'].isin(['Pass', 'Fail', 'Pass w/ Conditions']) &
    (df_join['License #'] > 0)
]

# Map each license_number -> its category (most-frequent category if multiple records)
lic_cat_map = (
    food_lic.dropna(subset=['license_number'])
    .groupby('license_number')['Category']
    .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else 'Other')
    .to_dict()
)

df_join['Category'] = df_join['License #'].map(lic_cat_map)
df_matched = df_join.dropna(subset=['Category']).copy()
print(f"Matched inspections: {df_matched.shape[0]:,} of {df_join.shape[0]:,}")

outcome_by_cat = (
    df_matched.groupby(['Category', 'Results']).size().reset_index(name='Count')
)
total_by_cat = (
    df_matched.groupby('Category').size().reset_index(name='Total')
)
outcome_by_cat = outcome_by_cat.merge(total_by_cat, on='Category')
outcome_by_cat['Percentage'] = (outcome_by_cat['Count'] / outcome_by_cat['Total'] * 100).round(1)

# Drop categories with very few inspections (keeps statistical reliability
# but lets in newer/smaller categories like Pop-Up Food)
outcome_by_cat = outcome_by_cat[outcome_by_cat['Total'] >= 30]

# ---- 6. outcomes_by_category.csv ----
outcome_by_cat.to_csv('outcomes_by_category.csv', index=False)
print(f"✓ outcomes_by_category.csv: {outcome_by_cat.shape}")
print(outcome_by_cat)

print("\nAll 6 CSVs written to current directory.")
print("Move them into projects/chicago-food-inspections/data/ in your repo.")

Loading Food Inspections dataset (~323 MB, may take a minute)...
Raw shape: (309344, 17)
Filtered shape: (246990, 18)
✓ driver_data.csv: (24, 5)
✓ yearly_pass.csv: (128, 4)
✓ before_after.csv: (8, 4)
✓ yearly_all_results.csv: (384, 5)

Querying Business Licenses dataset (server-side filtered)...
Food-related license rows downloaded: (208871, 4)
✓ coverage_gap.csv: (16, 5)

Joining inspections to licenses by License Number for Option 5...
Matched inspections: 215,004 of 260,689
✓ outcomes_by_category.csv: (12, 5)
                                      Category             Results   Count  \
0                 Mobile Food (trucks & carts)                Fail     541   
1                 Mobile Food (trucks & carts)                Pass    1392   
2                 Mobile Food (trucks & carts)  Pass w/ Conditions     221   
4   Retail Food (restaurants, grocery, bakery)                Fail   47291   
5   Retail Food (restaurants, grocery, bakery)                Pass  124770   
6   Retail Foo

In [3]:
"""
build_charts.py — Generates standalone interactive HTML chart files.
Run after build_data.py.
Output: 5 HTML files (4 main + 1 alternative for comparison).
"""
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

driver_data       = pd.read_csv('driver_data.csv')
yearly_pass       = pd.read_csv('yearly_pass.csv')
before_after      = pd.read_csv('before_after.csv')
coverage          = pd.read_csv('coverage_gap.csv')
outcomes_by_cat   = pd.read_csv('outcomes_by_category.csv')

facility_order = [
    'Restaurant', 'Grocery Store', 'School', "Children's Services Facility",
    'Bakery', 'Daycare Above and Under 2 Years', 'Daycare (2 - 6 Years)', 'Long Term Care'
]
facility_colors = alt.Scale(
    domain=facility_order,
    range=['#4E79A7', '#F28E2B', '#59A14F', '#E15759',
           '#76B7B2', '#EDC948', '#B07AA1', '#FF9DA7']
)

# ============================================================
# CHART 1 — CENTRAL VIZ: Small multiples + linked annotated timeline
# ============================================================
year_brush = alt.selection_interval(encodings=['x'])

small_mults = (
    alt.Chart(yearly_pass)
    .mark_line(strokeWidth=2.5, point=alt.OverlayMarkDef(size=40))
    .encode(
        x=alt.X('Year:O', title=None,
                axis=alt.Axis(labelAngle=-45, labelFontSize=8, labelOverlap=False)),
        y=alt.Y('Pass Rate:Q', title='Pass Rate (%)',
                scale=alt.Scale(domain=[0, 100])),
        color=alt.Color('Facility Type:N', scale=facility_colors, legend=None),
        opacity=alt.condition(year_brush, alt.value(1.0), alt.value(0.25)),
        tooltip=[
            alt.Tooltip('Facility Type:N'),
            alt.Tooltip('Year:O'),
            alt.Tooltip('Pass Rate:Q', format='.1f'),
            alt.Tooltip('Total:Q', title='Inspections', format=',')
        ]
    )
    .properties(width=280, height=180)
    .facet(
        facet=alt.Facet('Facility Type:N', title=None,
                        header=alt.Header(labelFontSize=11, labelFontWeight='bold')),
        columns=4
    )
    .resolve_scale(y='shared')
)

agg = (
    yearly_pass.assign(weighted=yearly_pass['Pass Rate'] * yearly_pass['Total'])
    .groupby('Year', as_index=False)
    .agg(weighted_sum=('weighted', 'sum'), total_sum=('Total', 'sum'))
)
agg['Pass Rate'] = agg['weighted_sum'] / agg['total_sum']
agg = agg[['Year', 'Pass Rate']]

base_overview = alt.Chart(agg).encode(
    x=alt.X('Year:O', title='Year (drag on this chart to filter the panels above)'),
    y=alt.Y('Pass Rate:Q', title='Avg Pass Rate (%)', scale=alt.Scale(domain=[0, 100]))
)
overview_line = base_overview.mark_line(color='#2c3e50', strokeWidth=3, point=True)

annotations = pd.DataFrame([
    {'Year': 2018, 'label': '2018: New CDPH violation rules', 'y_pos': 92},
    {'Year': 2020, 'label': '2020: COVID-19', 'y_pos': 78}
])
ann_rules = alt.Chart(annotations).mark_rule(
    color='#c0392b', strokeDash=[4, 4], strokeWidth=1.5
).encode(x='Year:O')
ann_text = alt.Chart(annotations).mark_text(
    align='left', dx=6, color='#c0392b', fontSize=11, fontWeight='bold'
).encode(x='Year:O', y=alt.Y('y_pos:Q'), text='label:N')

overview = (overview_line + ann_rules + ann_text).add_params(year_brush).properties(
    width=1240, height=180,
    title=alt.TitleParams(
        text='Citywide Average Pass Rate (2010–2025)',
        subtitle='Drag to highlight a year range — the panels above will dim outside your selection',
        fontSize=14
    )
)

central_viz = alt.vconcat(small_mults, overview, spacing=15).properties(
    title=alt.TitleParams(
        text='Pass Rates Across 8 Chicago Facility Types, 2010–2025',
        subtitle='Each panel shows one facility type. Hover for details. Drag the bottom timeline to focus on a year range.',
        fontSize=18, anchor='start', color='#2c3e50'
    )
).configure_view(stroke=None).configure_axis(
    labelFontSize=10, titleFontSize=11, grid=True, gridColor='#EEE'
)
central_viz.save('main_dashboard.html')
print("✓ main_dashboard.html")

# ============================================================
# CHART 2 — CONTEXTUAL VIZ #1: Dumbbell chart (2017 vs 2019)
# ============================================================
ba_long = before_after.melt(
    id_vars=['Facility Type', 'Drop'],
    value_vars=['Pass_2017', 'Pass_2019'],
    var_name='Year', value_name='Pass Rate'
)
ba_long['Year'] = ba_long['Year'].map({'Pass_2017': '2017', 'Pass_2019': '2019'})

bars = alt.Chart(before_after).mark_bar(height=4, color='#bbb').encode(
    y=alt.Y('Facility Type:N', sort=alt.SortField('Drop', order='descending'), title=None),
    x=alt.X('Pass_2019:Q', title='Pass Rate (%)', scale=alt.Scale(domain=[0, 100])),
    x2='Pass_2017:Q'
)
dots = alt.Chart(ba_long).mark_circle(size=200, opacity=1).encode(
    y=alt.Y('Facility Type:N', sort=alt.SortField('Drop', order='descending'), title=None),
    x=alt.X('Pass Rate:Q'),
    color=alt.Color('Year:N',
                    scale=alt.Scale(domain=['2017', '2019'], range=['#27ae60', '#c0392b']),
                    legend=alt.Legend(title='Year', orient='top')),
    tooltip=['Facility Type:N', 'Year:N', alt.Tooltip('Pass Rate:Q', format='.1f')]
)
labels = alt.Chart(before_after).mark_text(
    align='left', dx=8, fontSize=11, fontWeight='bold', color='#c0392b'
).encode(
    y=alt.Y('Facility Type:N', sort=alt.SortField('Drop', order='descending')),
    x='Pass_2017:Q',
    text=alt.Text('Drop:Q', format='.1f')
)

dumbbell = (bars + dots + labels).properties(
    width=900, height=320,
    title=alt.TitleParams(
        text='Two Years, One Rule Change',
        subtitle='Pass rates collapsed across every facility type between 2017 and 2019. Numbers show percentage-point drop.',
        fontSize=16, anchor='start', color='#2c3e50'
    )
).configure_view(stroke=None).configure_axis(grid=True, gridColor='#EEE')
dumbbell.save('before_after.html')
print("✓ before_after.html")

# ============================================================
# CHART 3 — CONTEXTUAL VIZ #2: Heatmap (Facility × Year)
# ============================================================
heatmap = alt.Chart(yearly_pass).mark_rect().encode(
    x=alt.X('Year:O', title='Year'),
    y=alt.Y('Facility Type:N', sort=facility_order, title=None),
    color=alt.Color('Pass Rate:Q',
                    scale=alt.Scale(scheme='redyellowgreen', domain=[0, 100]),
                    legend=alt.Legend(title='Pass Rate (%)', orient='right')),
    tooltip=[
        alt.Tooltip('Facility Type:N'),
        alt.Tooltip('Year:O'),
        alt.Tooltip('Pass Rate:Q', format='.1f'),
        alt.Tooltip('Total:Q', title='Inspections', format=',')
    ]
).properties(
    width=1000, height=320,
    title=alt.TitleParams(
        text='Where the Failures Concentrated',
        subtitle='Red = low pass rate, green = high. The 2018–2019 column lights up red across nearly every row.',
        fontSize=16, anchor='start', color='#2c3e50'
    )
)
heatmap_text = alt.Chart(yearly_pass).mark_text(fontSize=9).encode(
    x='Year:O',
    y=alt.Y('Facility Type:N', sort=facility_order),
    text=alt.Text('Pass Rate:Q', format='.0f'),
    color=alt.condition(
        'datum["Pass Rate"] < 35 || datum["Pass Rate"] > 75',
        alt.value('white'), alt.value('black')
    )
)
heatmap_full = (heatmap + heatmap_text).configure_view(stroke=None).configure_axis(
    labelFontSize=11, titleFontSize=12, labelLimit=200
)
heatmap_full.save('heatmap.html')
print("✓ heatmap.html")

# ============================================================
# CHART 4 — OPTION 4: Coverage gap as bars (gap below 100%)
# Single-direction red bars showing how far below full coverage each year fell
# ============================================================
coverage['Coverage_capped'] = coverage['Coverage %'].clip(upper=100)
coverage['Gap_below_100']   = (100 - coverage['Coverage_capped']).round(1)

gap_bars = alt.Chart(coverage).mark_bar(color='#c0392b').encode(
    x=alt.X('Year:O', title='Year', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('Gap_below_100:Q',
            title='Coverage gap (percentage points below 100%)',
            scale=alt.Scale(domain=[0, 25])),
    tooltip=[
        alt.Tooltip('Year:O'),
        alt.Tooltip('Active Licenses:Q', format=','),
        alt.Tooltip('Establishments Inspected:Q', format=','),
        alt.Tooltip('Uninspected:Q', title='Uninspected', format=','),
        alt.Tooltip('Gap_below_100:Q', title='Gap (pp below 100%)', format='.1f')
    ]
)

gap_highlight = alt.Chart(coverage[coverage['Year'].isin([2020, 2021])]).mark_bar(
    color='#7d1f1f'
).encode(x='Year:O', y='Gap_below_100:Q')

gap_labels = alt.Chart(coverage[coverage['Gap_below_100'] > 0]).mark_text(
    align='center', baseline='bottom', dy=-4,
    fontSize=10, fontWeight='bold', color='#333'
).encode(
    x='Year:O', y='Gap_below_100:Q',
    text=alt.Text('Gap_below_100:Q', format='.1f')
)

gap_chart = alt.layer(gap_bars, gap_highlight, gap_labels).properties(
    width=1000, height=420,
    title=alt.TitleParams(
        text='How Far Below Full Coverage? The Annual Inspection Gap, 2010–2025',
        subtitle=('Each bar shows how many percentage points coverage fell below 100%. '
                  'Darker red = COVID years (2020–21).'),
        fontSize=15, anchor='start', color='#2c3e50',
        subtitleFontSize=11, subtitleColor='#666'
    )
).configure_view(stroke=None).configure_axis(
    grid=True, gridColor='#EEE', labelFontSize=11, titleFontSize=12
)
gap_chart.save('coverage_gap.html')
print("✓ coverage_gap.html (Option 4: gap-from-100% bars)")

# ============================================================
# CHART 5 — OPTION 5: Inspection outcomes by license category
# Stacked horizontal bars showing pass/fail/PwC mix per license type
# ============================================================
cat_totals = outcomes_by_cat.groupby('Category')['Total'].first().sort_values(ascending=False)
cat_sorted = cat_totals.index.tolist()

result_palette = alt.Scale(
    domain=['Pass', 'Pass w/ Conditions', 'Fail'],
    range=['#27ae60', '#f39c12', '#c0392b']
)

outcomes_chart = alt.Chart(outcomes_by_cat).mark_bar().encode(
    y=alt.Y('Category:N', sort=cat_sorted, title=None,
            axis=alt.Axis(labelLimit=300, labelFontSize=11)),
    x=alt.X('Percentage:Q',
            title='% of inspections',
            scale=alt.Scale(domain=[0, 100])),
    color=alt.Color('Results:N', scale=result_palette,
                    legend=alt.Legend(title=None, orient='top')),
    order=alt.Order('Results:N', sort='ascending'),
    tooltip=[
        alt.Tooltip('Category:N'),
        alt.Tooltip('Results:N'),
        alt.Tooltip('Count:Q', format=','),
        alt.Tooltip('Percentage:Q', format='.1f', title='% of category')
    ]
).properties(
    width=900, height=380,
    title=alt.TitleParams(
        text='Pass, Fail, or Conditional? Outcomes by License Type',
        subtitle=('Each bar shows how 100% of inspections for that license category broke down. '
                  'Categories with fewer than 100 inspections are excluded.'),
        fontSize=15, anchor='start', color='#2c3e50',
        subtitleFontSize=11, subtitleColor='#666'
    )
).configure_view(stroke=None).configure_axis(
    grid=True, gridColor='#EEE', labelFontSize=11, titleFontSize=12
)
outcomes_chart.save('outcomes_by_category.html')
print("✓ outcomes_by_category.html (Option 5: outcomes by license type)")

print("\nDone — 5 HTML files written.")
print("Open coverage_gap.html (Option 4) and outcomes_by_category.html (Option 5)")
print("side by side, then decide which to keep as Chart 4 in your final article.")

✓ main_dashboard.html
✓ before_after.html
✓ heatmap.html
✓ coverage_gap.html (Option 4: gap-from-100% bars)
✓ outcomes_by_category.html (Option 5: outcomes by license type)

Done — 5 HTML files written.
Open coverage_gap.html (Option 4) and outcomes_by_category.html (Option 5)
side by side, then decide which to keep as Chart 4 in your final article.
